In [1]:
# Parameters cell
# thermodynamic integration vars
ti_parallel_n_jobs=10
ti_parallel_n_verbose=5
ti_dcount=50

# wbic mcmc settings
wbic_n_tune=2500
wbic_n_draws=1000
wbic_n_chains=1

# wbic block settings
wbic_block_size=1
wbic_parallel_n_jobs=1
wbic_parallel_n_verbose=10

In [ ]:
import os

n_components = 2
n_trials=100
regimes = [50, 250, 5000]
print(f"Running on a machine with {os.cput_count()} cpu cores")

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from pathlib import Path
from os import environ
import os

ROOT = Path(f"{Path.cwd().parents[1]}/outputs")
basedir = Path(f"{ROOT}/mixture/binom2d")
datadir = Path(f"{basedir}/data")
outputdir = Path(f"{basedir}/wbic-bias")

if not outputdir.exists():
  outputdir.mkdir(exist_ok=True)
  print(f"Created {outputdir}!")

print(f"Using datadir={datadir}")
print(f"Using outputdir={outputdir}")

dry = False # whether or not to save results

Using datadir=/Users/ashrafahmed/workspace/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixture/binom2d/data
Using outputdir=/Users/ashrafahmed/workspace/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixture/binom2d/wbic-bias


In [4]:
import pandas as pd

dgps_file = f"{datadir}/dgp.csv"
dgps=pd.read_csv(dgps_file, index_col=0).reset_index()
dgps.head()

,dsid,p0,p1,w0,w1
0,regular,0.15,0.35,0.75,0.25
1,e-singular,0.25,0.35,0.75,0.25
2,singular1,0.15,0.35,1.00,0.00
3,singular2,0.35,0.35,0.75,0.25


In [5]:
from sklearn_extensions.mixbinom import BinomialMixture
import numpy as np


def find_truth_by_dsid(dsid: str):
  dgp = dgps.query(f"dsid=='{dsid}'")
  truth = dgp[["p0", "p1", "w0", "w1"]].iloc[0].tolist()
  return truth

# def rlct_by_dsid(dsid: str):
#   truth = find_truth_by_dsid(dsid)
#   n_components = np.ceil(len(truth)/2)
#   rlct = None
#   match dsid:
#     case "regular" | "e-singular":
#       rlct = (n_components*2-1)/2
#     case "singular1" | "singular2":
#       rlct = 1
#     case _:
#       raise Exception(f"Uknown dsid={dsid}")

#   return rlct

# def approx_free_energy_by_dsid(dsid, n_trials, X):
#   n=len(X)
  
#   average_log_likelihood = None
#   model = BinomialMixture(n_components=2, n_trials=n_trials, enforce_ordering=False)
#   input_data = np.column_stack([X, np.full_like(X, n_trials)])
#   model.fit(input_data)
#   mle, _ = model.point_estimate()
#   log_p = mixbinom.logpmf(weights=[mle[2], 1-mle[2]], probs=[mle[0], mle[1]], n=n_trials, x=X) # sample likelihood under the mle
#   average_log_likelihood = log_p.mean()

#   afe = -n*average_log_likelihood+rlct_by_dsid(dsid)*np.log(n)
#   return afe


# def expected_free_energy_by_dsid(dsid, n_trials, n):
#   X = np.arange(0, n_trials + 1)  # 0 to n_trials inclusive
#   truth = find_truth_by_dsid(dsid)
  
#   probs = truth[0:2]    # [p0, p1]
#   weights = truth[2:4]  # [w0, w1]
  
#   log_q = mixbinom.logpmf(n=n_trials, weights=weights, probs=probs, x=X)
  
#   expected_log_likelihood = np.sum(np.exp(log_q) * log_q)
  
#   second_order_term = rlct_by_dsid(dsid) * np.log(n)
#   efe = -n * expected_log_likelihood + second_order_term
  
#   return efe

In [6]:
# compute AFE and WBIC for many samples, 
# AFE requires sampling with beta=1 while WBIC requires sampling with beta=1/sqrt(n)
# for different n

In [7]:
import pandas as pd
import json
from pathlib import Path

results_file = Path(f"{outputdir}/fe_estimators_results.csv")
if dry:
  results_file = Path(f"{results_file}.dry")

# Load existing results if file exists
if results_file.exists():
  fe_estimators_df = pd.read_csv(results_file)
  # Create a set of completed (run, regime, dsid) tuples for fast lookup
  completed = set(
    zip(fe_estimators_df["trial"], fe_estimators_df["regime"], fe_estimators_df["dsid"])
  )
  fe_estimators_data = fe_estimators_df.to_dict("records")
else:
  completed = set()
  fe_estimators_data = []

def save_results():
  pd.DataFrame(fe_estimators_data).to_csv(results_file, index=False)

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
from itertools import product
import time
from pymc_extensions.tempered_mixbinom import TemperedBinomialMixture, free_energy, free_energy_parallel
from pymc_extensions import pmx
from scipy_extensions import mixbinom
from tqdm.notebook import tqdm
import pymc as pm
import numpy as np
import arviz as az

def run_single_dsid(dsid, run, regime, datadir, n_trials, n_draws, n_tune, n_chains):
  dataset = pd.read_csv(f"{datadir}/{dsid}-{regime}.csv")
  X = dataset.iloc[:, run].to_numpy()
  n_obs = len(X)
  
  fe = free_energy_parallel(n_trials=n_trials, 
                            X=X,
                            betas=np.linspace(0,1,ti_dcount)**2,
                            n_components=n_components,
                            parallel_n_jobs=ti_parallel_n_jobs,
                            parallel_n_verbose=ti_parallel_n_verbose,
                            nuts_sampler="nutpie" if not dry else "numpyro")
  
  with TemperedBinomialMixture(X=X, n_trials=100, beta=1/np.log(n_obs)) as model:
    idata = model.sample(
      draws=n_draws,
      tune=n_tune,
      chains=n_chains,
      progressbar=False,
      nuts_sampler="nutpie" if not dry else "numpyro",
      cores=1
    )
    
    diverging = idata.sample_stats.diverging.values
    divs_per_chain = diverging.sum(axis=1)

    wbic = model.wbic(idata)

    print(f"run={run}, n_components={n_components}, regime={regime}, dsid={dsid}, wbic={wbic:.4f}, fe={fe:.4f}")
    
    return {
      "dsid": dsid,
      "regime": regime,
      "n": regime,
      "trial": run,
      "wbic": wbic,
      "fe": fe,
      "chains": n_chains,
      "draws": n_draws,
      "tune": n_tune,
      "mean_divergences": divs_per_chain.mean(),
      "total_divergences": diverging.sum(),
      "max_divergences": divs_per_chain.max(),
      "divergences_per_chain": divs_per_chain.tolist(),
      "chain_tree_depth": idata.sample_stats.depth.values.max()
    }

n_cores = os.cpu_count()
# All dsids
all_dsids = dgps["dsid"].unique()

# Group tasks by regime
# if n_cores > len(all_dsids):
#   block_size = int(n_cores/len(all_dsids)-1)
# else:
# block_size = 1

# Main loop - one run at a time, parallelize all (regime, dsid) combos
for block_start in tqdm(range(0, 1000, block_size), desc="blocks"):
  block_runs = range(block_start, min(block_start + wbic_block_size, 1000))
  for regime in regimes:
    start = time.perf_counter()
    # Build list of all (regime, dsid) pairs that haven't been completed
    tasks_to_run = [
      (run, dsid) 
      for run in block_runs
      for dsid in all_dsids
      if (run, regime, dsid) not in completed
    ]
    
    if not tasks_to_run:
      print(f"skipped {tasks_to_run}")
      continue
    
    # Run all regime x dsid combinations in parallel
    results = Parallel(n_jobs=wbic_parallel_n_jobs, verbose=wbic_parallel_verbose)(
      delayed(run_single_dsid)(
        dsid, run, regime, datadir, n_trials, n_draws, n_tune, n_chains
      )
      for run, dsid in tasks_to_run
    )
    
    # Collect results
    for result in results:
      fe_estimators_data.append(result)
      completed.add((result["trial"], result["regime"], result["dsid"]))
  
    end = time.perf_counter()
    # Save after each run completes
    save_results()
    if not dry:
      !git add "../../outputs/mixture/binom2d/wbic-bias/fe_estimators_results.csv"
      !git commit -m f"run n_components={n_components} block_runs={block_runs} complete"
      !git push

blocks:   0%|          | 0/1000 [00:00<?, ?it/s]

skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []
skipped []


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  42 out of  50 | elapsed:  3.0min remaining:   34.0s
[Parallel(n_jobs=10)]: Done  50 out of  50 | elapsed:  3.0min finished
Exception ignored on calling ctypes callback function: <function ExecutionEngine._raw_object_cache_notify at 0x13d7b0fe0>
Traceback (most recent call last):
  File "/Users/ashrafahmed/workspace/waterloo-slt-reading-group/zoo/python/.venv/lib/python3.12/site-packages/llvmlite/binding/executionengine.py", line 178, in _raw_object_cache_notify
    def _raw_object_cache_notify(self, data):

KeyboardInterrupt: 


run=6, n_components=2, regime=250, dsid=regular, wbic=799.6898, fe=802.0601


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:  3.3min
[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  42 out of  50 | elapsed:  2.7min remaining:   30.8s
[Parallel(n_jobs=10)]: Done  50 out of  50 | elapsed:  2.7min finished


...

run=6, n_components=2, regime=250, dsid=singular1, wbic=655.3923, fe=657.2730


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  42 out of  50 | elapsed:  6.6min remaining:  1.3min
[Parallel(n_jobs=10)]: Done  50 out of  50 | elapsed:  6.7min finished


run=6, n_components=2, regime=250, dsid=singular2, wbic=741.6251, fe=742.9166


[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed: 17.9min
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed: 17.9min finished


[mixbinom2d-wbic3 3434e85] frun range(6, 7) complete
 1 file changed, 9 insertions(+), 5 deletions(-)
Enter passphrase for key '/Users/ashrafahmed/.ssh/id_rsa': 